In [1]:
# Initial Python environment setup...
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from math import radians, sin, cos, sqrt, atan2
import os # reading the input files we have access to

print(os.listdir('../input'))

['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
# load the train data
#train_df =  pd.read_csv('../input/new-york-city-taxi-fare-prediction/train.csv', nrows = 10_000_000)
train =  pd.read_csv('../input/short-train/short_train.csv')#, nrows = 10_000_000)
train.head()

FileNotFoundError: [Errno 2] No such file or directory: '../input/short-train/short_train.csv'

In [3]:
# load test data
test = pd.read_csv('../input/new-york-city-taxi-fare-prediction/test.csv')
test.head()

,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2010-10-01 21:26:11.0000001,2010-10-01 21:26:11 UTC,-73.983130,40.761970,-73.994386,40.749236,1
1,2013-10-06 01:38:00.00000083,2013-10-06 01:38:00 UTC,-73.948505,40.753977,-73.808195,40.731952,2
2,2012-03-30 19:13:53.0000001,2012-03-30 19:13:53 UTC,-73.973964,40.791979,-73.979018,40.785544,1
3,2012-02-08 02:57:23.0000001,2012-02-08 02:57:23 UTC,-73.991478,40.738907,-73.907198,40.861572,2
4,2013-12-13 22:56:00.000000237,2013-12-13 22:56:00 UTC,-73.986281,40.740067,-73.933927,40.856781,2


In [4]:
# remove all missing values from train data
train.dropna(inplace=True)
# remove all missing values from test data
test.dropna(inplace=True)

NameError: name 'train' is not defined

In [5]:
# A function to calculate the distance:
# Haversine distance calculation
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points
    on the earth specified in radians.
    """
    # Radius of the Earth in km
    R = 6371.0

    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c

    return distance

In [6]:
# Feature Engineering for the training set
# Calculate distance using Haversine formula
train['distance'] = haversine_distance(train['pickup_latitude'], train['pickup_longitude'], train['dropoff_latitude'], train['dropoff_longitude'])
# Convert pickup datetime to datetime object
train['pickup_datetime'] = pd.to_datetime(train['pickup_datetime'])
# Extract features from pickup datetime
train['day_of_week'] = train['pickup_datetime'].dt.dayofweek
train['month'] = train['pickup_datetime'].dt.month
train['hour'] = train['pickup_datetime'].dt.hour

train.head(100)

NameError: name 'train' is not defined

# linear model

In [7]:
# Define features and target
X = train[['distance', 'day_of_week', 'month', 'hour']]
y = train['fare_amount']

# Split the data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Train linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on the validation set
y_pred_val = model.predict(X_val)

# Evaluate the model's performance on the validation set
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
print("Validation RMSE:", rmse_val)

NameError: name 'train' is not defined

predictions on the given test set:

In [8]:
# Once satisfied with the model's performance, train the model on the entire original training set
model.fit(X, y)

NameError: name 'model' is not defined

In [9]:
# Feature Engineering for the training set
test['distance'] = haversine_distance(test['pickup_latitude'], test['pickup_longitude'], test['dropoff_latitude'], test['dropoff_longitude'])
test['pickup_datetime'] = pd.to_datetime(test['pickup_datetime'])
test['day_of_week'] = test['pickup_datetime'].dt.dayofweek
test['month'] = test['pickup_datetime'].dt.month
test['hour'] = test['pickup_datetime'].dt.hour

# Define features for the test set
X_test = test[['distance', 'day_of_week', 'month', 'hour']]

# Make predictions on the test set
y_pred_test = model.predict(X_test)
# Add the predictions to the test dataframe
test['fare_amount'] = y_pred_test

submission = pd.DataFrame(
    {'key': test['key'], 'fare_amount': test['fare_amount']},
    columns = ['key', 'fare_amount'])

# save the predictions to a CSV file
submission.to_csv('submission.csv', index = False)



NameError: name 'model' is not defined